In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import*
from pyspark.sql.window import*
from pyspark.sql.functions import*

spark=SparkSession.builder.appName('demo').getOrCreate()

In [0]:
# Preferred to save time and prevent from wrong datatype
# Manually giving the datatype(inferschema =false)
# inferSchema=True automatically detects column data types.But it takes extra read time and may infer incorrect data types if the data is inconsistent.
# Eg: numeric column with one text value may be inferred as StringType
data = [
    ('Renu', 99, 'coimbatore'),
    ('Ram', 98, 'chennai'),
    (None,7,None)
]

#value = ['Name', 'Mark', 'location']

value=StructType([
    StructField('name',StringType(),True),
    StructField('Mark',IntegerType(),False),
    StructField('Subject',StringType(),True)
])

df = spark.createDataFrame(data, schema=value)

df.display()

name,Mark,Subject
Renu,99,coimbatore
Ram,98,chennai
null,7,null


In [0]:
#group by

data = [
    (1, "A", "IT", 5000),
    (2, "B", "IT", 7000),
    (3, "C", "IT", 5000),
    (4, "D", "HR", 4000),
    (5, "E", "HR", 6000),
    (6, "F", "Finance", 8000),
    (7, "G", "Finance", 8000),
    (8, "H", "Finance", 10000),
]

columns = ["EmpID", "EmpName", "Department", "salary"]

df = spark.createDataFrame(data, columns)

df_grp = df.groupBy("Department").agg(
    sum("salary").alias("sum_sal"),
    avg("salary").alias("avg_sal"),
    countDistinct("EmpID").alias("emp_count")
).orderBy(col("sum_sal").desc())

df_grp.display()

Department,sum_sal,avg_sal,emp_count
Finance,26000,8666.666666666666,3
IT,17000,5666.666666666667,3
HR,10000,5000.0,2


In [0]:
#windows


data = [
    (1, "Arun", "IT", 50000),
    (2, "Bala", "IT", 60000),
    (3, "Chitra", "IT", None),
    (4, "Divya", "HR", 45000),
    (5, "Eshan", "HR", 45000),
    (6, "Farah", "HR", 50000),
    (7, "Gokul", "Finance", 70000),
    (8, "Hari", "Finance", None),
    (9, "Ishita", "Finance", 70000)
]

columns = ["Id", "Name", "Department", "Salary"]

df = spark.createDataFrame(data, columns)
df.display()

window_spec = Window.partitionBy("Department").orderBy("Salary")

df1 = df.withColumn("Lead_Salary", lead("Salary",2).over(window_spec)) \
        .withColumn("Lag_Salary", lag("Salary").over(window_spec)) \
        .withColumn("Rank", rank().over(window_spec)) \
        .withColumn("DenseRank", dense_rank().over(window_spec)) \
        .withColumn("RowNumber", row_number().over(window_spec)) \
        .withColumn("IsNull_Salary", col("Salary").isNull()) \
        .withColumn("IsNotNull_Salary", col("Salary").isNotNull())#.filter(col('Lead_Salary').isNull())# we use isnull/not is null to filter the records like below
        

df1.show()

Id,Name,Department,Salary
1,Arun,IT,50000
2,Bala,IT,60000
3,Chitra,IT,null
4,Divya,HR,45000
5,Eshan,HR,45000
6,Farah,HR,50000
7,Gokul,Finance,70000
8,Hari,Finance,null
9,Ishita,Finance,70000


+---+------+----------+------+-----------+----------+----+---------+---------+-------------+----------------+
| Id|  Name|Department|Salary|Lead_Salary|Lag_Salary|Rank|DenseRank|RowNumber|IsNull_Salary|IsNotNull_Salary|
+---+------+----------+------+-----------+----------+----+---------+---------+-------------+----------------+
|  8|  Hari|   Finance|  NULL|      70000|      NULL|   1|        1|        1|         true|           false|
|  7| Gokul|   Finance| 70000|       NULL|      NULL|   2|        2|        2|        false|            true|
|  9|Ishita|   Finance| 70000|       NULL|     70000|   2|        2|        3|        false|            true|
|  4| Divya|        HR| 45000|      50000|      NULL|   1|        1|        1|        false|            true|
|  5| Eshan|        HR| 45000|       NULL|     45000|   1|        1|        2|        false|            true|
|  6| Farah|        HR| 50000|       NULL|     45000|   3|        2|        3|        false|            true|
|  3|Chitr

In [0]:
#Windows fn 

data = [
    ("HR", 1000),
    ("HR", 1200),
    ("HR", 1500),
    ("HR", 1800),
    ("HR", 2000),
    ("IT", 1100),
    ("IT", 1300),
    ("IT", 1600),
    ("IT", 1900),
    ("IT", 2200)
]

columns = ["Department", "salary"]

df = spark.createDataFrame(data, columns)

win_cumulative = (
    Window
    .partitionBy("Department")
    .orderBy(col("salary"))
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

# Last 3 rows (rolling window)
win_last_3 = (
    Window
    .partitionBy("Department")
    .orderBy(col("salary"))
    .rowsBetween(-2, Window.currentRow)
)

# Current + next 2 rows (forward window)
win_forward = (
    Window
    .partitionBy("Department")
    .orderBy(col("salary"))
    .rowsBetween(0, 2)
)

# Previous 2 + current row
win_prev2 = (
    Window
    .partitionBy("Department")
    .orderBy(col("salary"))
    .rowsBetween(-2, 0)
)

###########################################

df_result = df \
    .withColumn("running_salary", sum("salary").over(win_cumulative)) \
    .withColumn("last_3_sum", sum("salary").over(win_last_3)) \
    .withColumn("forward_3_sum", sum("salary").over(win_forward)) \
    .withColumn("prev2_sum", sum("salary").over(win_prev2))

df_result.show()

+----------+------+--------------+----------+-------------+---------+
|Department|salary|running_salary|last_3_sum|forward_3_sum|prev2_sum|
+----------+------+--------------+----------+-------------+---------+
|        HR|  1000|          1000|      1000|         3700|     1000|
|        HR|  1200|          2200|      2200|         4500|     2200|
|        HR|  1500|          3700|      3700|         5300|     3700|
|        HR|  1800|          5500|      4500|         3800|     4500|
|        HR|  2000|          7500|      5300|         2000|     5300|
|        IT|  1100|          1100|      1100|         4000|     1100|
|        IT|  1300|          2400|      2400|         4800|     2400|
|        IT|  1600|          4000|      4000|         5700|     4000|
|        IT|  1900|          5900|      4800|         4100|     4800|
|        IT|  2200|          8100|      5700|         2200|     5700|
+----------+------+--------------+----------+-------------+---------+



In [0]:
#case stmt /between

data = [
    (1, "A", 45000),
    (2, "B", 47000),
    (3, "C", 49000),
    (4, "D", 50000),
    (5, "E", 52000),
    (6, "F", 60000)
]

columns = ["EmpID", "EmpName", "salary"]

df = spark.createDataFrame(data, columns)

df.display()

#df_val = df.withColumn("comments",
#    when(col("salary") < 47000, "low")
#    .when((col("salary") >= 47000) & (col("salary") <= 50000), "medium")
#    .otherwise("high")
#)

### OR

df_val = df.withColumn("comments",
    when(col("salary") < 47000, "low")
    .when(col("salary").between(47000, 50000), "medium")
    .otherwise("high")
)

df_val.display()

EmpID,EmpName,salary
1,A,45000
2,B,47000
3,C,49000
4,D,50000
5,E,52000
6,F,60000


EmpID,EmpName,salary,comments
1,A,45000,low
2,B,47000,medium
3,C,49000,medium
4,D,50000,medium
5,E,52000,high
6,F,60000,high


In [0]:
#DROP duplicate records,fill random values in null value

data = [
    (1, "A", "IBM", "India", 50000),
    (2, "B", "IBM", None, 60000),
    (3, "C", None, "USA", None),
    (4, "D", "TCS", "India", 70000),
    (4, "D", "TCS", "India", 70000),   # duplicate row
    (5, "E", None, None, 80000),
]

columns = ["EmpID", "EmpName", "Company", "Country", "Salary"]

df = spark.createDataFrame(data, columns)

df.display()

EmpID,EmpName,Company,Country,Salary
1,A,IBM,India,50000
2,B,IBM,null,60000
3,C,null,USA,null
4,D,TCS,India,70000
4,D,TCS,India,70000
5,E,null,null,80000


In [0]:
#Drop NA

df_any = df.dropna(how="any")
df_any.display()

df_subset = df.dropna(subset=["Company", "Country"])
df_subset.display()

EmpID,EmpName,Company,Country,Salary
1,A,IBM,India,50000
4,D,TCS,India,70000
4,D,TCS,India,70000


EmpID,EmpName,Company,Country,Salary
1,A,IBM,India,50000
4,D,TCS,India,70000
4,D,TCS,India,70000


In [0]:
#Fill NA

df_mean = df.select(mean(col("Salary"))).first()[0] #same way we can calcute avg,sum
df_fill = df.fillna({"Salary": df_mean,"Country": "loc"})
df_fill.display()

#Harcode the specific value in all empty place irrespective of columns
df_fill_emp = df.fillna("empty")
df_fill_emp.display()

EmpID,EmpName,Company,Country,Salary
1,A,IBM,India,50000
2,B,IBM,loc,60000
3,C,null,USA,66000
4,D,TCS,India,70000
4,D,TCS,India,70000
5,E,null,loc,80000


EmpID,EmpName,Company,Country,Salary
1,A,IBM,India,50000
2,B,IBM,empty,60000
3,C,empty,USA,null
4,D,TCS,India,70000
4,D,TCS,India,70000
5,E,empty,empty,80000


In [0]:
#Drop the dupicate records

df_distinct = df.distinct()
df_distinct.display()

df_dup_all = df.dropDuplicates()
df_dup_all.display()

# Based on specific columns 
df_dup_subset = df.dropDuplicates(["EmpID", "Company"])
df_dup_subset.display()

EmpID,EmpName,Company,Country,Salary
1,A,IBM,India,50000
2,B,IBM,null,60000
3,C,null,USA,null
4,D,TCS,India,70000
5,E,null,null,80000


EmpID,EmpName,Company,Country,Salary
1,A,IBM,India,50000
2,B,IBM,null,60000
3,C,null,USA,null
4,D,TCS,India,70000
5,E,null,null,80000


EmpID,EmpName,Company,Country,Salary
1,A,IBM,India,50000
2,B,IBM,null,60000
3,C,null,USA,null
4,D,TCS,India,70000
5,E,null,null,80000


In [0]:
#Pivot and Stack(Transpose)
data = [
    ('Accenture', 'IT', 50000),
    ('Accenture', 'HR', 30000),
    ('Accenture', 'Finance', 40000),
    ('CTS', 'IT', 45000),
    ('CTS', 'HR', 25000),
    ('Infosys', 'IT', 48000),
    ('Infosys', 'Finance', 42000),
    ('TCS', 'IT', 47000),
    ('TCS', 'HR', 26000)
]

columns = ['Company', 'Department', 'Salary']

df_pivot = spark.createDataFrame(data, columns)

df_pivot.show()

# ---------------- PIVOT ----------------
df_p = df_pivot.groupBy("Company") \
    .pivot("Department").agg(sum("Salary"))

df_p.show()

#  STACK 
df_stack = df_p.selectExpr(
    "Company",
    "stack(3, 'Finance', Finance, 'HR', HR, 'IT', IT) as (Department, Salary)"
)

df_stack.show()

+---------+----------+------+
|  Company|Department|Salary|
+---------+----------+------+
|Accenture|        IT| 50000|
|Accenture|        HR| 30000|
|Accenture|   Finance| 40000|
|      CTS|        IT| 45000|
|      CTS|        HR| 25000|
|  Infosys|        IT| 48000|
|  Infosys|   Finance| 42000|
|      TCS|        IT| 47000|
|      TCS|        HR| 26000|
+---------+----------+------+

+---------+-------+-----+-----+
|  Company|Finance|   HR|   IT|
+---------+-------+-----+-----+
|Accenture|  40000|30000|50000|
|      CTS|   NULL|25000|45000|
|  Infosys|  42000| NULL|48000|
|      TCS|   NULL|26000|47000|
+---------+-------+-----+-----+

+---------+----------+------+
|  Company|Department|Salary|
+---------+----------+------+
|Accenture|   Finance| 40000|
|Accenture|        HR| 30000|
|Accenture|        IT| 50000|
|      CTS|   Finance|  NULL|
|      CTS|        HR| 25000|
|      CTS|        IT| 45000|
|  Infosys|   Finance| 42000|
|  Infosys|        HR|  NULL|
|  Infosys|        IT|

In [0]:
#collectlist and Flatten

data = [
    ("A", [1, 2, 3]),
    ("A", [4]),
    ("B", [5, 6]),
    ("B", [1]),
    ("C", None)
]

schema = StructType([
    StructField("key", StringType(), True),
    StructField("values", ArrayType(IntegerType()), True)
])

df = spark.createDataFrame(data, schema)

df_grp=df.groupBy(col('key')).agg(collect_list(col('values')))
df_grp.display()

df_group=df.groupBy(col('key')).agg(flatten(collect_list(col('values'))))
df_group.display()

key,collect_list(values)
A,"List(List(1, 2, 3), List(4))"
B,"List(List(5, 6), List(1))"
C,List()


key,flatten(collect_list(values))
A,"List(1, 2, 3, 4)"
B,"List(5, 6, 1)"
C,List()


In [0]:
# Exlode outer-->it will inclue null records as well

df_explode=df.withColumn('exp',explode_outer(col('values')))
df_explode.display()

df_collect=df_explode.groupby('key').agg(collect_list(col('exp')))
df_collect.display()

key,values,exp
A,"List(1, 2, 3)",1
A,"List(1, 2, 3)",2
A,"List(1, 2, 3)",3
A,List(4),4
B,"List(5, 6)",5
B,"List(5, 6)",6
B,List(1),1
C,null,null


key,collect_list(exp)
A,"List(1, 2, 3, 4)"
B,"List(5, 6, 1)"
C,List()


In [0]:
# AND and OR ,collect_list

data = [
    ("Rahul", 85, 1),
    ("Adarsh", 73, 1),
    ("Riti", 95, 1),
    ("Viraj", 80, 1),
    ("Vimal", 83, 2),
    ("Neha", 77, 2),
    ("Priti", 73, 2),
    ("Himanshi", 85, 2)
]

columns = ["passenger_name", "weight_kg", "lift_id"]

df = spark.createDataFrame(data, columns)
df.show()

df_win=Window.partitionBy(col('lift_id')).orderBy(col('weight_kg'))
df_result=df.withColumn('running_total',sum('weight_kg').over(df_win))\
    .filter(
        ((col("lift_id") == 1) & (col("running_total") <= 300)) | ((col("lift_id") == 2) & (col("running_total") <= 350))
    )
df_result.display()

df_group=df_result.groupBy(col('lift_id')).agg(concat_ws(',',collect_list(col('passenger_name'))))
df_group.display()

+--------------+---------+-------+
|passenger_name|weight_kg|lift_id|
+--------------+---------+-------+
|         Rahul|       85|      1|
|        Adarsh|       73|      1|
|          Riti|       95|      1|
|         Viraj|       80|      1|
|         Vimal|       83|      2|
|          Neha|       77|      2|
|         Priti|       73|      2|
|      Himanshi|       85|      2|
+--------------+---------+-------+



passenger_name,weight_kg,lift_id,running_total
Adarsh,73,1,73
Viraj,80,1,153
Rahul,85,1,238
Priti,73,2,73
Neha,77,2,150
Vimal,83,2,233
Himanshi,85,2,318


lift_id,"concat_ws(,, collect_list(passenger_name))"
1,"Adarsh,Viraj,Rahul"
2,"Priti,Neha,Vimal,Himanshi"


In [0]:
# Array sort will sort the products in specific column like this(Camera,Laptop,TV)

data = [
    ("Electronics", "TV"),
    ("Electronics", "Laptop"),
    ("Electronics", "Camera"),
    ("Grocery", "Rice"),
    ("Grocery", "Wheat"),
    ("Grocery", "Sugar")
]

columns = ["Category", "ProductName"]

df = spark.createDataFrame(data, columns)

result_df = df.groupBy("Category").agg(
        concat_ws(
            ",",
            array_sort(collect_list("ProductName"))
        ).alias("Products")
    )

result_df.show(truncate=False)

+-----------+----------------+
|Category   |Products        |
+-----------+----------------+
|Electronics|Camera,Laptop,TV|
|Grocery    |Rice,Sugar,Wheat|
+-----------+----------------+



In [0]:
#join,union

products_data = [
    (1, "iPhone", "Apple"),
    (2, "Galaxy", "Samsung"),
    (3, "Pixel", "Google"),
    (4, "Redmi", "Xiaomi")
]

products_cols = ["ProductID", "ProductName", "Brand"]
products_df = spark.createDataFrame(products_data, products_cols)


orders_data = [
    (101, 1, 2),
    (102, 2, 1),
    (103, 3, 5),
    (104, 1, 1),
    (105, 4, 3)
]

orders_cols = ["OrderID", "ProductID", "Quantity"]
orders_df = spark.createDataFrame(orders_data, orders_cols)

new_products_data = [
    (5, "OnePlus", "OnePlus"),
    (6, "iPad", "Apple")
]

products_cols = ["ProductID", "ProductName", "Brand"]
new_products_df = spark.createDataFrame(new_products_data, products_cols)




joined_df = orders_df.join(products_df, "ProductID", "inner")

filtered_df = products_df.filter(
    col("Brand").isin("Apple", "Samsung")
)

union_df = products_df.union(new_products_df)

joined_df.display()
filtered_df.display()
union_df.display()


ProductID,OrderID,Quantity,ProductName,Brand
1,104,1,iPhone,Apple
1,101,2,iPhone,Apple
2,102,1,Galaxy,Samsung
3,103,5,Pixel,Google
4,105,3,Redmi,Xiaomi


ProductID,ProductName,Brand
1,iPhone,Apple
2,Galaxy,Samsung


ProductID,ProductName,Brand
1,iPhone,Apple
2,Galaxy,Samsung
3,Pixel,Google
4,Redmi,Xiaomi
5,OnePlus,OnePlus
6,iPad,Apple


In [0]:
#union by name

data1 = [
    (1, "Apple", "iPhone"),
    (2, "Samsung", "Galaxy")]

df1 = spark.createDataFrame(data1, ["ProductID", "Brand", "ProductName"])
df1.show()

data2 = [
    ("OnePlus", 3, "Nord"),
    ("Google", 4, "Pixel")]

df2 = spark.createDataFrame(data2,["Brand", "ProductID", "ProductName"]  # different order
)
df2.show()

final_df = df1.unionByName(df2)
final_df.display()


+---------+-------+-----------+
|ProductID|  Brand|ProductName|
+---------+-------+-----------+
|        1|  Apple|     iPhone|
|        2|Samsung|     Galaxy|
+---------+-------+-----------+

+-------+---------+-----------+
|  Brand|ProductID|ProductName|
+-------+---------+-----------+
|OnePlus|        3|       Nord|
| Google|        4|      Pixel|
+-------+---------+-----------+



ProductID,Brand,ProductName
1,Apple,iPhone
2,Samsung,Galaxy
3,OnePlus,Nord
4,Google,Pixel


# Interview questions
###IBM

In [0]:
#Need firstname,Middle name,lastname(Asked in IBM 1 st round)
spark = SparkSession.builder.appName("NameSplit").getOrCreate()

data = [
    ("ABC XYZ",),
    ("ABC PQR XYZ",)
]

df = spark.createDataFrame(data, ['name'])
df.display()

df_name=df.withColumn('firstname',split('name',' ').getItem(0))\
    .withColumn('middle_name',when(size(split('name',' ')) ==3,split('name',' ').getItem(1)).otherwise(lit('none')))\
        .withColumn('last_name',when(size(split('name',' ')) ==3,split('name',' ').getItem(2)).otherwise(split('name',' ').getItem(1)))

df_name.display()

name
ABC XYZ
ABC PQR XYZ


name,firstname,middle_name,last_name
ABC XYZ,ABC,none,XYZ
ABC PQR XYZ,ABC,PQR,XYZ


In [0]:
#Find the customer with the highest number of orders in each city.(Asked in IBM 1 st round)
data = [
    (1, 101, "Amit", "Mumbai", 2000),
    (2, 101, "Amit", "Mumbai", 3000),
    (3, 102, "Priya", "Mumbai", 2500),
    (4, 103, "Rahul", "Pune", 4000),
    (5, 103, "Rahul", "Pune", 3500),
    (6, 104, "Sneha", "Pune", 1500),
    (7, 105, "Karan", "Bangalore", 5000),
    (8, 105, "Karan", "Bangalore", 6000),
    (9, 105, "Karan", "Bangalore", 2000)
]

columns = ["order_id", "customer_id", "customer_name", "city", "amount"]
df = spark.createDataFrame(data, columns)
df.display()

df_count = df.groupBy("city", "customer_name")\
    .agg(count("*").alias("total_count"))

win = Window.partitionBy("city").orderBy(desc("total_count"))

result = df_count.withColumn("rank", dense_rank().over(win))

result.display()

order_id,customer_id,customer_name,city,amount
1,101,Amit,Mumbai,2000
2,101,Amit,Mumbai,3000
3,102,Priya,Mumbai,2500
4,103,Rahul,Pune,4000
5,103,Rahul,Pune,3500
6,104,Sneha,Pune,1500
7,105,Karan,Bangalore,5000
8,105,Karan,Bangalore,6000
9,105,Karan,Bangalore,2000


+---------+-------------+-----------+----+
|     city|customer_name|total_count|rank|
+---------+-------------+-----------+----+
|Bangalore|        Karan|          3|   1|
|   Mumbai|         Amit|          2|   1|
|   Mumbai|        Priya|          1|   2|
|     Pune|        Rahul|          2|   1|
|     Pune|        Sneha|          1|   2|
+---------+-------------+-----------+----+



In [0]:
# give top company based on their total revenue(Asked in IBM 2 nd round)

#calculate the difference between the current year revenue and previous year revenue in sql(using the below dataset) (Asked in IBM 2 nd round)-->use window fn(lag)

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("CompanyRevenue").getOrCreate()

data = [
    ("IBM", 2021, 9000),
    ("IBM", 2020, 7000),
    ("ACN", 2020, 7000),
    ("ACN", 2019, 5000),
    ("INFY", 2023, 7000),
    ("INFY", 2024, 9000)
]

columns = ["CO_NAME", "YEAR", "REVENUE"]
df = spark.createDataFrame(data, columns)
df.display()


df_grp = df.groupBy("CO_NAME").agg(sum("REVENUE").alias("total_revenue"))
win = Window.orderBy(desc("total_revenue"))
df_rank = df_grp.withColumn("dense_rank",dense_rank().over(win)).filter(col('dense_rank')==1)
df_rank.display()

CO_NAME,YEAR,REVENUE
IBM,2021,9000
IBM,2020,7000
ACN,2020,7000
ACN,2019,5000
INFY,2023,7000
INFY,2024,9000


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------+-------------+----------+
|CO_NAME|total_revenue|dense_rank|
+-------+-------------+----------+
|    IBM|        16000|         1|
|   INFY|        16000|         1|
+-------+-------------+----------+



In [0]:
#Flatten the structure and how many records will get in an output (Asked in IBM 2 nd round)

data = [
    {
        "order_id": 101,
        "customer": {
            "name": "Alice",
            "email": "alice@example.com"
        },
        "items": [
            {"product": "Laptop", "qty": 1, "price": 75000},
            {"product": "Mouse", "qty": 2, "price": 500}
        ],
        "payment": {
            "method": "UPI",
            "status": "Success"
        }
    },
    {
        "order_id": 102,
        "customer": {
            "name": "Bob",
            "email": "bob@example.com"
        },
        "items": [
            {"product": "Keyboard", "qty": 1, "price": 1200}
        ],
        "payment": {
            "method": "Card",
            "status": "Pending"
        }
    }
]

df = spark.createDataFrame(data)
df.display()

df_exploded = df.withColumn("item", explode(col("items")))

df_result = df_exploded.select(
    col("order_id"),
    col("customer.name").alias("customer_name"),
    col("customer.email").alias("cust_email"),
    col("item.product").alias("item_product"),
    col("item.qty").alias("quantity"),
    col("item.price").alias("price"),
    col("payment.method").alias("pay_method"),
    col("payment.status").alias("pay_status")
)

df_result.display()

customer,items,order_id,payment
"Map(name -> Alice, email -> alice@example.com)","List(Map(product -> Laptop, qty -> 1, price -> 75000), Map(product -> Mouse, qty -> 2, price -> 500))",101,"Map(method -> UPI, status -> Success)"
"Map(name -> Bob, email -> bob@example.com)","List(Map(product -> Keyboard, qty -> 1, price -> 1200))",102,"Map(method -> Card, status -> Pending)"


order_id,customer_name,cust_email,item_product,quantity,price,pay_method,pay_status
101,Alice,alice@example.com,Laptop,1,75000,UPI,Success
101,Alice,alice@example.com,Mouse,2,500,UPI,Success
102,Bob,bob@example.com,Keyboard,1,1200,Card,Pending


###EXL

In [0]:
Orders-->| order_id | item_id | price | address_id |
Address-->| address_id | city |

ques 1:Find the number of orders per city in sql

ques 2: For each order, find the top 3 most expensive items using price

ques 3:
Table A --> a a d b c c
Table B --> NULL NULL D B A A
Give the result for all 4 joins(right,inner,left,outer)

Pyspark
1.Fillna
2.group by ques
3.Both pyspark and sql -->Find the hrs or each employee
Emp_name	Type	date_time
Emp A	Login	07-04-2026 07:00
Emp A	Logout	07-04-2026 12:00
Emp A	Login	07-04-2026 14:00
Emp B	Login	07-04-2026 08:00
Emp C	Login	07-04-2026 07:00
Emp C	Logout	07-04-2026 12:00

###PWC Total sales by region where amount > 10000

In [0]:

df_result = df \
    .filter(col("sales") > 10000) \
    .groupBy("region") \
    .agg(sum("sales").alias("total_sales"))

df_result.display()

In [0]:
#3rd Highest Salary IN SQL

WITH cte AS (
    SELECT *,
           ROW_NUMBER() OVER (ORDER BY salary DESC) AS rn
    FROM employee
)
SELECT *
FROM cte
WHERE rn = 3;

In [0]:
data = [
    (1, "flight1", "Delhi", "Hyderabad"),
    (1, "flight2", "Hyderabad", "Kochi"),
    (1, "flight3", "Kochi", "Mangalore"),
    (2, "flight1", "Mumbai", "Ayodhya"),
    (2, "flight2", "Ayodhya", "Gorakhpur")
]

columns = ["cust_id", "flight_id", "origin", "destination"]

df = spark.createDataFrame(data, columns)
df.show()

w = Window.partitionBy("cust_id").orderBy("flight_id")

df_window = df.withColumn("first_origin", first("origin").over(w)) \
              .withColumn("last_destination", last("destination").over(w))

result = df_window.groupBy("cust_id") \
                  .agg(
                      first("first_origin").alias("origin"),
                      last("last_destination").alias("destination")
                  )

result.show()


+-------+---------+---------+-----------+
|cust_id|flight_id|   origin|destination|
+-------+---------+---------+-----------+
|      1|  flight1|    Delhi|  Hyderabad|
|      1|  flight2|Hyderabad|      Kochi|
|      1|  flight3|    Kochi|  Mangalore|
|      2|  flight1|   Mumbai|    Ayodhya|
|      2|  flight2|  Ayodhya|  Gorakhpur|
+-------+---------+---------+-----------+

+-------+------+-----------+
|cust_id|origin|destination|
+-------+------+-----------+
|      1| Delhi|  Mangalore|
|      2|Mumbai|  Gorakhpur|
+-------+------+-----------+



In [0]:
# SALTING(Not much imporant for interview ,they ask the definition).Used during data skewness

SALT_BUCKETS = 3
new_df_salted = new_df.withColumn("salt",floor(rand() * SALT_BUCKETS))#gives the random values for each row

existing_df_salted = existing_df.withColumn("salt",explode(array([lit(i) for i in range(SALT_BUCKETS)])))#small dataset give copy of each records 3 time becoz we mentioned sat_ucket is 3 

joined_df = new_df_salted.join(existing_df_salted,on=["Customer_ID", "salt"],how="left")



In [0]:
##write and read format

#---------- READ FORMATS ----------

# CSV
df_csv = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/mnt/data/input.csv")

# PARQUET
df_parquet = spark.read.format("parquet").load("/mnt/data/input_parquet")

# JSON
df_json = spark.read.format("json") \
    .option("multiline", "true") \
    .load("/mnt/data/input.json")

# AVRO
df_avro = spark.read.format("avro").load("/mnt/data/input_avro")

# ORC
df_orc = spark.read.format("orc").load("/mnt/data/input_orc")

# TEXT
df_text = spark.read.format("text").load("/mnt/data/input.txt")

# DELTA (file)
df_delta_file = spark.read.format("delta").load("/mnt/data/input_delta")

# TABLE READS
df_table = spark.read.table("default.customer_table")
df_table_sql = spark.sql("SELECT * FROM default.customer_table")

# ---------- WRITE FORMATS ----------overwrite/append

# CSV
df_csv.write.format("csv") \
    .option("header", "true") \
    .mode("overwrite") \
    .save("/mnt/data/output_csv")

# PARQUET
df_csv.write.format("parquet") \
    .mode("overwrite") \
    .save("/mnt/data/output_parquet")

# JSON
df_csv.write.format("json") \
    .mode("overwrite") \
    .save("/mnt/data/output_json")

# AVRO
df_csv.write.format("avro") \
    .mode("overwrite") \
    .save("/mnt/data/output_avro")

# ORC
df_csv.write.format("orc") \
    .mode("overwrite") \
    .save("/mnt/data/output_orc")

# TEXT
df_csv.selectExpr("CAST(col1 AS STRING)") \
    .write.format("text") \
    .mode("overwrite") \
    .save("/mnt/data/output_text")

# DELTA (file)
df_csv.write.format("delta") \
    .mode("overwrite") \
    .save("/mnt/data/output_delta")

# ---------- TABLE WRITES ----------

# Managed table
df_csv.write.mode("overwrite").saveAsTable("default.customer_managed")

# Delta table
df_csv.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("default.customer_delta")

# Insert into existing table
df_csv.write.mode("append").insertInto("default.customer_delta")

# ---------- WRITE MODES ----------
# overwrite | append | ignore | error / errorifexists

print("All read & write formats executed in a single cell.")


In [0]:
#Read modes(Helps to drop/create the new column for invalid records)

path = "data.csv"  # replace with your file path

print("=== PERMISSIVE MODE ===")
df_permissive = (
    spark.read
    .option("header", "true")
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .csv(path)
)
df_permissive.show(truncate=False)

print("Corrupt rows (PERMISSIVE):")
df_permissive.filter("_corrupt_record IS NOT NULL").show(truncate=False)


print("=== DROPMALFORMED MODE ===")
df_drop = (
    spark.read
    .option("header", "true")
    .option("mode", "DROPMALFORMED")
    .csv(path)
)
df_drop.show(truncate=False)


print("=== FAILFAST MODE ===")
try:
    df_failfast = (
        spark.read
        .option("header", "true")
        .option("mode", "FAILFAST")
        .csv(path)
    )
    df_failfast.show(truncate=False)
except Exception as e:
    print("FAILFAST error:")
    print(e)


### **DATE FUNCTIONS**

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, to_date, to_timestamp, date_format, datediff, date_add, date_sub,
    add_months, months_between, year, month, dayofmonth, hour, minute, second,
    current_date, current_timestamp, trunc, next_day, weekofyear, unix_timestamp,
    from_unixtime, expr, lit
)

# Initialize Spark
spark = SparkSession.builder.appName("DateFunctionsSeparateDFs").getOrCreate()

# Base DataFrame with sample data
data = [
    ("2023-01-10", "2023-01-15 13:45:30"),
    ("2023-02-20", "2023-02-25 09:15:00"),
    ("2023-03-05", "2023-03-10 22:30:45")
]
columns = ["date_str", "timestamp_str"]
df = spark.createDataFrame(data, columns)
df.display()



date_str,timestamp_str
2023-01-10,2023-01-15 13:45:30
2023-02-20,2023-02-25 09:15:00
2023-03-05,2023-03-10 22:30:45


In [0]:
# ---- 1. to_date ----
df_to_date = df.select("date_str", to_date(col("date_str"), "yyyy-MM-dd").alias("date"))
df_to_date.display()

# ---- 2. to_timestamp ----
df_to_timestamp = df.select("timestamp_str", to_timestamp(col("timestamp_str"), "yyyy-MM-dd HH:mm:ss").alias("timestamp"))
df_to_timestamp.display()

# ---- 3. date_format ----
df_date_format = df.select("date_str", date_format(to_date(col("date_str")), "dd/MM/yyyy").alias("formatted_date"))
df_date_format.display()

# ---- 4. datediff ----
df_datediff = df.select(
    "date_str",
    datediff(current_date(), to_date(col("date_str"))).alias("days_diff")
)
df_datediff.display()

# ---- 5. date_add ----
df_date_add = df.select("date_str", date_add(to_date(col("date_str")), 10).alias("date_plus_10"))
df_date_add.display()

# ---- 6. date_sub ----
df_date_sub = df.select("date_str", date_sub(to_date(col("date_str")), 5).alias("date_minus_5"))
df_date_sub.display()

# ---- 7. add_months ----
df_add_months = df.select("date_str", add_months(to_date(col("date_str")), 2).alias("date_plus_2_months"))
df_add_months.display()

# ---- 8. months_between ----
df_months_between = df.select("date_str", months_between(current_date(), to_date(col("date_str"))).alias("months_between"))
df_months_between.display()

# ---- 9. Extract parts ----
df_extract_parts = df.select(
    "date_str",
    year(to_date(col("date_str"))).alias("year"),
    month(to_date(col("date_str"))).alias("month"),
    dayofmonth(to_date(col("date_str"))).alias("day"),
    hour(to_timestamp(col("timestamp_str"))).alias("hour"),
    minute(to_timestamp(col("timestamp_str"))).alias("minute"),
    second(to_timestamp(col("timestamp_str"))).alias("second")
)
df_extract_parts.display()

# ---- 10. current_date and current_timestamp ----
df_current = df.select(current_date().alias("current_date"), current_timestamp().alias("current_timestamp"))
df_current.display()

# ---- 11. trunc ----
df_trunc = df.select(
    "date_str",
    trunc(to_date(col("date_str")), "year").alias("year_start"),
    trunc(to_date(col("date_str")), "month").alias("month_start")
)
df_trunc.display()

# ---- 12. next_day ----
df_next_day = df.select("date_str", next_day(to_date(col("date_str")), "Sun").alias("next_sunday"))
df_next_day.display()

# ---- 13. weekofyear ----
df_weekofyear = df.select("date_str", weekofyear(to_date(col("date_str"))).alias("week_of_year"))
df_weekofyear.display()

# ---- 14. unix_timestamp and from_unixtime ----
df_unix = df.select(
    "timestamp_str",
    unix_timestamp(col("timestamp_str"), "yyyy-MM-dd HH:mm:ss").alias("unix_time"),
    from_unixtime(unix_timestamp(col("timestamp_str"), "yyyy-MM-dd HH:mm:ss")).alias("from_unix_time")
)
df_unix.display()

# ---- 15. expr date arithmetic ----
df_expr = df.select(
    "date_str",
    expr("date_add(to_date(date_str), 7)").alias("date_plus_7"),
    expr("date_sub(to_date(date_str), 3)").alias("date_minus_3")
)
df_expr.display()


date_str,date
2023-01-10,2023-01-10
2023-02-20,2023-02-20
2023-03-05,2023-03-05


timestamp_str,timestamp
2023-01-15 13:45:30,2023-01-15T13:45:30.000Z
2023-02-25 09:15:00,2023-02-25T09:15:00.000Z
2023-03-10 22:30:45,2023-03-10T22:30:45.000Z


date_str,formatted_date
2023-01-10,10/01/2023
2023-02-20,20/02/2023
2023-03-05,05/03/2023


date_str,days_diff
2023-01-10,1265
2023-02-20,1224
2023-03-05,1211


date_str,date_plus_10
2023-01-10,2023-01-20
2023-02-20,2023-03-02
2023-03-05,2023-03-15


...

date_str,year_start,month_start
2023-01-10,2023-01-01,2023-01-01
2023-02-20,2023-01-01,2023-02-01
2023-03-05,2023-01-01,2023-03-01


date_str,next_sunday
2023-01-10,2023-01-15
2023-02-20,2023-02-26
2023-03-05,2023-03-12


date_str,week_of_year
2023-01-10,2
2023-02-20,8
2023-03-05,9


timestamp_str,unix_time,from_unix_time
2023-01-15 13:45:30,1673790330,2023-01-15 13:45:30
2023-02-25 09:15:00,1677316500,2023-02-25 09:15:00
2023-03-10 22:30:45,1678487445,2023-03-10 22:30:45


date_str,date_plus_7,date_minus_3
2023-01-10,2023-01-17,2023-01-07
2023-02-20,2023-02-27,2023-02-17
2023-03-05,2023-03-12,2023-03-02


SCD types -->MERGE into(Very important for interview)

Interview ques for spark and Databricks-->Raja's Data Engineering(1 playlist -144 videos)

Hadoop/Map Reduce/Spark architecture -->Gowtham SB (2 playlist around 15+ hrs)

Good To learn
1.Autoloader-->including(Schema hint,2 modes-file listing)
2.Checkpointing
3.collect-->This action will nor prefered in development it leads to OOM issue because it retrive the data from all worker node and gives to our Master node at the same time
4.optimize+z-ordering vs Liquid clustering
5.CDC VS Change Data Feed(Table level in delta table)
6.Broadcast varible
7.Types of cluster-->job nd Interactive 
8.Saprk nd databricks Architecture(databricks is built on top of spark)


sql important topics
1.Group by,partition,joins windows fn(Dense_rank),cte
